# 🌍 USGS Global Earthquake Monitor — Kafka + Elasticsearch
**Source:** USGS Earthquake Hazards Program API (free, no key — earthquake.usgs.gov)
**Stack:** USGS API → Kafka → PySpark → Elasticsearch (full-text + geospatial) → Kibana
**Orchestration:** Apache Airflow

## Overview
Real-time streaming pipeline monitoring global earthquake activity from the USGS
Earthquake API. Events are streamed via Kafka, enriched with PySpark (magnitude
classification, region mapping, tsunami risk scoring), and indexed into Elasticsearch
for full-text search, geospatial queries, and Kibana dashboards.

**API:** https://earthquake.usgs.gov/fdsnws/event/1/query — completely free, no key
**Data:** All global earthquakes M≥1.0, updated every 5 minutes
**Volume:** ~10,000–15,000 earthquakes/month worldwide

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests kafka-python pyspark elasticsearch pandas

import requests
import json
import time
import logging
import pandas as pd
from datetime import datetime, timezone, timedelta, date
from kafka import KafkaProducer
from elasticsearch import Elasticsearch, helpers
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, to_timestamp, when, lit,
    current_timestamp, udf, abs as spark_abs
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, BooleanType
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('EarthquakePipeline')
print('✅ All imports loaded')

In [ ]:
# USGS Earthquake API — completely free, no API key
USGS_BASE_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query'

# Kafka
KAFKA_BOOTSTRAP    = 'localhost:9092'
KAFKA_TOPIC_EQ     = 'usgs-earthquakes'

# Elasticsearch
ES_HOST            = 'http://localhost:9200'
ES_INDEX_RAW       = 'earthquakes-raw'
ES_INDEX_ENRICHED  = 'earthquakes-enriched'

# Magnitude classification thresholds (USGS standard)
MAG_CLASSES = [
    (8.0, 'Great'),
    (7.0, 'Major'),
    (6.0, 'Strong'),
    (5.0, 'Moderate'),
    (4.0, 'Light'),
    (3.0, 'Minor'),
    (0.0, 'Micro')
]

print('✅ Config ready')
print(f'   USGS API: {USGS_BASE_URL}')
print(f'   Elasticsearch: {ES_HOST}')

## Section 2 — USGS API Producer → Kafka

In [ ]:
def classify_magnitude(mag: float) -> str:
    for threshold, label in MAG_CLASSES:
        if mag >= threshold:
            return label
    return 'Micro'

def compute_tsunami_risk(magnitude: float, depth_km: float, lat: float) -> str:
    """
    Simplified tsunami risk: shallow + strong + coastal = high risk.
    Coastal = lat between -60 and 60 (tropical/temperate)
    """
    if magnitude >= 7.5 and depth_km <= 70 and abs(lat) <= 60:
        return 'high'
    if magnitude >= 6.5 and depth_km <= 100:
        return 'medium'
    return 'low'

def fetch_earthquakes(start_time: str, end_time: str,
                       min_magnitude: float = 1.0) -> list:
    """
    Fetch earthquake events from USGS API.
    Returns GeoJSON FeatureCollection.
    """
    params = {
        'format':       'geojson',
        'starttime':    start_time,
        'endtime':      end_time,
        'minmagnitude': min_magnitude,
        'orderby':      'time',
        'limit':        20000
    }
    resp = requests.get(USGS_BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    features = resp.json().get('features', [])
    records  = []
    now = datetime.now(timezone.utc).isoformat()

    for f in features:
        props = f.get('properties', {})
        geom  = f.get('geometry', {})
        coords = geom.get('coordinates', [None, None, None])
        lon, lat, depth = coords[0], coords[1], coords[2]
        mag   = props.get('mag') or 0.0
        time_ms = props.get('time', 0)
        eq_time = datetime.fromtimestamp(time_ms / 1000, tz=timezone.utc).isoformat()

        records.append({
            'event_id':         f.get('id', ''),
            'event_time':       eq_time,
            'magnitude':        float(mag),
            'magnitude_class':  classify_magnitude(float(mag)),
            'depth_km':         float(depth) if depth is not None else 0.0,
            'latitude':         float(lat)   if lat   is not None else 0.0,
            'longitude':        float(lon)   if lon   is not None else 0.0,
            'location':         {'lat': float(lat or 0), 'lon': float(lon or 0)},  # ES geo_point
            'place':            props.get('place', ''),
            'status':           props.get('status', ''),  # reviewed / automatic
            'tsunami_flag':     int(props.get('tsunami', 0)),
            'tsunami_risk':     compute_tsunami_risk(float(mag), float(depth or 0), float(lat or 0)),
            'alert':            props.get('alert', ''),  # green/yellow/orange/red
            'felt_reports':     int(props.get('felt', 0) or 0),
            'cdi':              float(props.get('cdi', 0) or 0),  # community intensity
            'mmi':              float(props.get('mmi', 0) or 0),  # max Mercalli intensity
            'net':              props.get('net', ''),
            'event_type':       props.get('type', 'earthquake'),
            'url':              props.get('url', ''),
            'sig':              int(props.get('sig', 0) or 0),  # significance 0–1000
            'nst':              int(props.get('nst', 0) or 0),  # station count
            'gap':              float(props.get('gap', 0) or 0), # azimuthal gap
            'dmin':             float(props.get('dmin', 0) or 0),
            'ingested_at':      now,
            'source':           'usgs-earthquake-api'
        })
    logger.info(f'✅ Fetched {len(records)} earthquakes ({start_time} → {end_time})')
    return records

def publish_earthquakes_to_kafka(hours_back: int = 24) -> int:
    end   = datetime.now(timezone.utc)
    start = end - timedelta(hours=hours_back)
    records = fetch_earthquakes(
        start.strftime('%Y-%m-%dT%H:%M:%S'),
        end.strftime('%Y-%m-%dT%H:%M:%S'),
        min_magnitude=1.0
    )
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    for r in records:
        producer.send(KAFKA_TOPIC_EQ, key=r['event_id'].encode(), value=r)
    producer.flush()
    producer.close()
    logger.info(f'✅ Published {len(records)} earthquakes to Kafka')
    return len(records)

# publish_earthquakes_to_kafka(hours_back=24)

## Section 3 — PySpark Enrichment → Elasticsearch Index

In [ ]:
spark = SparkSession.builder \
    .appName('EarthquakeEnrichment') \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,'
            'org.elasticsearch:elasticsearch-spark-30_2.12:8.13.0') \
    .config('es.nodes', 'localhost') \
    .config('es.port', '9200') \
    .config('es.nodes.wan.only', 'true') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

EQ_SCHEMA = StructType([
    StructField('event_id',        StringType(),  True),
    StructField('event_time',      StringType(),  True),
    StructField('magnitude',       DoubleType(),  True),
    StructField('magnitude_class', StringType(),  True),
    StructField('depth_km',        DoubleType(),  True),
    StructField('latitude',        DoubleType(),  True),
    StructField('longitude',       DoubleType(),  True),
    StructField('place',           StringType(),  True),
    StructField('status',          StringType(),  True),
    StructField('tsunami_flag',    IntegerType(), True),
    StructField('tsunami_risk',    StringType(),  True),
    StructField('alert',           StringType(),  True),
    StructField('felt_reports',    IntegerType(), True),
    StructField('sig',             IntegerType(), True),
    StructField('event_type',      StringType(),  True),
    StructField('ingested_at',     StringType(),  True),
])

def enrich_and_index_batch(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    enriched = batch_df \
        .withColumn('depth_category',
            when(col('depth_km') <= 70,  'shallow')
            .when(col('depth_km') <= 300, 'intermediate')
            .otherwise('deep')
        ) \
        .withColumn('alert_level',
            when(col('alert') == '', 'none')
            .otherwise(col('alert'))
        ) \
        .withColumn('is_significant', col('sig') >= 600) \
        .withColumn('is_felt', col('felt_reports') > 0) \
        .withColumn('processed_at', current_timestamp())

    # Write to Elasticsearch using bulk indexing
    docs = json.loads(enriched.toJSON().collect().__str__().replace("'",'"'))

    es = Elasticsearch(ES_HOST)
    actions = [
        {
            '_index': ES_INDEX_ENRICHED,
            '_id':    doc['event_id'],
            '_source': doc
        }
        for doc in [json.loads(r) for r in enriched.toJSON().collect()]
    ]
    success, failed = helpers.bulk(es, actions, raise_on_error=False)
    logger.info(f'⚡ Batch {batch_id}: {success} indexed, {failed} failed → Elasticsearch')

kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
    .option('subscribe', KAFKA_TOPIC_EQ) \
    .option('startingOffsets', 'latest') \
    .load()

parsed_df = kafka_df \
    .select(from_json(col('value').cast('string'), EQ_SCHEMA).alias('d')) \
    .select('d.*') \
    .withColumn('event_timestamp', to_timestamp(col('event_time')))

query = parsed_df.writeStream \
    .foreachBatch(enrich_and_index_batch) \
    .outputMode('append') \
    .trigger(processingTime='30 seconds') \
    .start()

print('🚀 Streaming → Elasticsearch started')
# query.awaitTermination()

## Section 4 — Elasticsearch Index Mapping + Queries

In [ ]:
ES_INDEX_MAPPING = {
    'mappings': {
        'properties': {
            'event_id':        {'type': 'keyword'},
            'event_time':      {'type': 'date'},
            'magnitude':       {'type': 'float'},
            'magnitude_class': {'type': 'keyword'},
            'depth_km':        {'type': 'float'},
            'location': {
                'type': 'geo_point'   # enables geospatial queries!
            },
            'place':           {'type': 'text', 'analyzer': 'standard'},
            'tsunami_risk':    {'type': 'keyword'},
            'alert_level':     {'type': 'keyword'},
            'depth_category':  {'type': 'keyword'},
            'sig':             {'type': 'integer'},
            'felt_reports':    {'type': 'integer'},
            'is_significant':  {'type': 'boolean'},
            'ingested_at':     {'type': 'date'},
        }
    },
    'settings': {
        'number_of_shards': 1,
        'number_of_replicas': 0,
        'refresh_interval': '5s'
    }
}

def setup_elasticsearch_index():
    es = Elasticsearch(ES_HOST)
    if not es.indices.exists(index=ES_INDEX_ENRICHED):
        es.indices.create(index=ES_INDEX_ENRICHED, body=ES_INDEX_MAPPING)
        logger.info(f'✅ Index created: {ES_INDEX_ENRICHED}')
    else:
        logger.info(f'ℹ️  Index already exists: {ES_INDEX_ENRICHED}')

def search_earthquakes_near_location(lat: float, lon: float, radius_km: float = 500) -> dict:
    """Geospatial query — find earthquakes within radius of a point."""
    es = Elasticsearch(ES_HOST)
    query = {
        'query': {
            'bool': {
                'must': [{'range': {'magnitude': {'gte': 3.0}}}],
                'filter': [{
                    'geo_distance': {
                        'distance': f'{radius_km}km',
                        'location': {'lat': lat, 'lon': lon}
                    }
                }]
            }
        },
        'sort': [{'magnitude': {'order': 'desc'}}],
        'size': 20
    }
    result = es.search(index=ES_INDEX_ENRICHED, body=query)
    hits = result['hits']['hits']
    print(f'\n🌍 Earthquakes M≥3.0 within {radius_km}km of ({lat},{lon}):')
    for h in hits[:5]:
        s = h['_source']
        print(f"  M{s['magnitude']} {s['magnitude_class']} — {s['place']}")
    return result

def get_significant_events_today() -> dict:
    """Full-text + filter query: significant earthquakes today."""
    es = Elasticsearch(ES_HOST)
    today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    query = {
        'query': {
            'bool': {
                'filter': [
                    {'term': {'is_significant': True}},
                    {'range': {'event_time': {'gte': today}}}
                ]
            }
        },
        'aggs': {
            'by_magnitude_class': {
                'terms': {'field': 'magnitude_class', 'size': 10}
            },
            'by_tsunami_risk': {
                'terms': {'field': 'tsunami_risk', 'size': 5}
            }
        },
        'size': 0
    }
    return es.search(index=ES_INDEX_ENRICHED, body=query)

# setup_elasticsearch_index()
# search_earthquakes_near_location(-6.2088, 106.8456, radius_km=1000)  # Near Jakarta

## Section 5 — Pipeline Summary

| Layer | Technology | Details |
|---|---|---|
| **Source** | USGS Earthquake API | Global M≥1.0 events, free, no key, updated every 5 min |
| **Ingest** | Apache Kafka | `usgs-earthquakes`, GZIP |
| **Processing** | PySpark Streaming | Enrichment: depth category, tsunami risk, alert level |
| **Search/Index** | **Elasticsearch** | geo_point mapping, full-text search, aggregations |
| **Dashboard** | Kibana | Geospatial heatmaps, magnitude timelines |
| **Orchestration** | Apache Airflow | Hourly fetch + index + quality alerts |

**Why Elasticsearch?** Geospatial queries (geo_distance, geo_bounding_box) enable
radius-based searches impossible in traditional SQL warehouses.
Full-text search on `place` field enables location name queries.

**Volume:** ~10,000–15,000 earthquake events/month, ~500/day globally